In [ ]:
import sys
sys.path.append("..")          # lets the notebook import our code from src/

import pandas as pd
from pathlib import Path

def load_pad_ufes_20(data_dir):
    root = Path(data_dir)
    csv_files = list(root.rglob("metadata.csv"))
    if not csv_files:
        raise FileNotFoundError(f"metadata.csv not found under {root}")

    df = pd.read_csv(csv_files[0])
    dx_col = next((c for c in ("dx", "diagnostic") if c in df.columns), None)
    img_col = next((c for c in ("img_id", "image_id") if c in df.columns), None)
    if dx_col is None or img_col is None:
        raise ValueError(f"Expected diagnosis and image ID columns; found {list(df.columns)}")

    df["dx"] = df[dx_col].astype(str).str.upper()
    image_files = {
        p.name: p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    }

    def find_image(image_id):
        name = Path(str(image_id)).name
        if name in image_files:
            return str(image_files[name])
        for suffix in (".jpg", ".jpeg", ".png"):
            if name + suffix in image_files:
                return str(image_files[name + suffix])
        return None

    df["image_path"] = df[img_col].map(find_image)
    df = df.dropna(subset=["image_path"]).copy()
    df["label"] = df["dx"].isin({"MEL", "BCC", "SCC"}).astype(int)
    return df
from src.train import get_device
from src.experiment import run_external

TAG = "quick_"                 # use "" later, for the full run
device = get_device()          # GPU or CPU

# Load PAD-UFES-20 with our four classes
df_ext = load_pad_ufes_20("../data/pad_ufes_20")
print(df_ext.shape)
print(df_ext["dx"].value_counts())              # images per diagnosis
print(df_ext["label"].value_counts())           # 0 = benign, 1 = malignant

ImportError: cannot import name 'load_pad_ufes_20' from 'src.data' (c:\Users\RARES\Documents\Research-Project-New\notebooks\..\src\data.py)